# Student Performance — Grade Prediction

**Goal:** Predict a student's final math grade (`G3`, scored 0-20) from personal, family, and lifestyle features — *without* using the intermediate grades `G1` and `G2`, which would leak the answer.

**Dataset:** [UCI Student Performance (via Kaggle)](https://www.kaggle.com/datasets/devansodariya/student-performance-data)  
395 students - 33 original columns - Portuguese secondary schools

---

## Notebook Structure

| # | Section | What it does |
|---|---------|-------------|
| 1 | **Imports** | Load all libraries once |
| 2 | **Load & Clean** | Read CSV, drop leaky/irrelevant columns |
| 3 | **EDA** | Distributions, scatter & box plots |
| 4 | **Correlation Heatmap** | See which features are related to G3 |
| 5 | **Preprocessing** | Encode categoricals, scale, train/test split |
| 6 | **Model Comparison** | Train 8 algorithms, print metrics table |
| 7 | **Model Picker** | **Choose which model to use — edit here** |
| 8 | **Overfitting Analysis** | CV gap table + learning curve |
| 9 | **Final Predictions** | Predict G3 on test set, show results |
| 10 | **Feature Importance** | Which features drive the prediction most |

> Every cell imports what it needs and re-trains if necessary, so you can run any cell independently without restarting the kernel.

## 1. Install & Import Libraries

We use:
- **pandas / numpy** — data manipulation
- **matplotlib / seaborn** — visualisation
- **scikit-learn** — all ML models, preprocessing, and evaluation metrics

Uncomment the `!pip install` line if you're running this for the first time.

In [ ]:
# !pip install scikit-learn pandas matplotlib seaborn

import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, cross_val_score, learning_curve, KFold)
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.metrics          import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model     import LinearRegression, Ridge, Lasso
from sklearn.tree             import DecisionTreeRegressor
from sklearn.ensemble         import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm              import SVR
from sklearn.neighbors        import KNeighborsRegressor

print('=' * 45)
print('  All libraries imported successfully')
print('=' * 45)

## 2. Load & Clean Dataset

The CSV may use a **comma** or **semicolon** separator, so we auto-detect it.

### Columns we drop and why

| Column | Reason |
|--------|--------|
| `G1` | 1st period grade — direct numerical proxy of G3. Including it causes **data leakage**. |
| `G2` | 2nd period grade — same issue, even stronger correlation with G3. |
| `school` | Binary school identifier. Not a generalisable predictor. |

After dropping these we have **29 features** to predict G3.

In [ ]:
import os, pandas as pd

def load_student_data(path='student_data.csv'):
    if not os.path.exists(path):
        url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/student-mat.csv'
        print(f'  File not found locally — downloading...')
        raw = pd.read_csv(url, sep=None, engine='python')
    else:
        with open(path, 'r') as f:
            first_line = f.readline()
        sep = ';' if ';' in first_line else ','
        raw = pd.read_csv(path, sep=sep)
    if raw.shape[1] == 1:
        raise ValueError('Only 1 column loaded — wrong separator.')
    return raw

df_raw = load_student_data('student_data.csv')
COLS_TO_DROP = ['G1', 'G2', 'school']
df = df_raw.drop(columns=COLS_TO_DROP)

print('=' * 45)
print('  Dataset Loaded')
print('=' * 45)
print(f'  Rows      : {df_raw.shape[0]}')
print(f'  Columns   : {df_raw.shape[1]} original -> {df.shape[1]} after drop')
print(f'  Features  : {df.shape[1] - 1}  +  1 target (G3)')
print(f'  Dropped   : {COLS_TO_DROP}')
print('=' * 45)
df.head()

## 3. Exploratory Data Analysis (EDA)

Before building any model, we need to **understand the data**:
- Are there missing values?
- What does the target (G3) look like? Any outliers?
- Which features seem related to G3?

The four plots show:
1. **G3 distribution** — shape of the target
2. **Absences vs G3** — do more absences mean lower grades?
3. **Study time vs G3** — does more study time raise grades?
4. **Past failures vs G3** — how strongly do past failures predict future ones?

In [ ]:
import numpy as np

missing = df.isnull().sum().sum()
print('=' * 45)
print('  Dataset Overview')
print('=' * 45)
print(f'  Shape          : {df.shape[0]} rows x {df.shape[1]} cols')
print(f'  Missing values : {missing} (none)' if missing == 0 else f'  Missing values : {missing}')
print()
print('  G3 (Final Grade) Statistics:')
print('  ' + '-' * 30)
stats = df['G3'].describe()
for k, v in stats.items():
    print(f'  {k:8s}: {v:.2f}')
print('=' * 45)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.01)

axes[0,0].hist(df['G3'], bins=21, color='steelblue', edgecolor='white')
axes[0,0].set_title('G3 (Final Grade) Distribution')
axes[0,0].set_xlabel('Grade (0-20)'); axes[0,0].set_ylabel('# Students')

axes[0,1].scatter(df['absences'], df['G3'], alpha=0.4, color='steelblue')
axes[0,1].set_title('Absences vs G3')
axes[0,1].set_xlabel('Number of absences'); axes[0,1].set_ylabel('Final Grade (G3)')

st_vals = sorted(df['studytime'].unique())
axes[1,0].boxplot([df[df['studytime']==s]['G3'].values for s in st_vals], labels=st_vals)
axes[1,0].set_title('Study Time vs G3')
axes[1,0].set_xlabel('Study time (1=<2h  2=2-5h  3=5-10h  4=>10h)')
axes[1,0].set_ylabel('Final Grade (G3)')

fail_vals = sorted(df['failures'].unique())
axes[1,1].boxplot([df[df['failures']==f]['G3'].values for f in fail_vals], labels=fail_vals)
axes[1,1].set_title('Past Failures vs G3')
axes[1,1].set_xlabel('Number of past failures'); axes[1,1].set_ylabel('Final Grade (G3)')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150); plt.show()
print('  Saved -> eda_plots.png')

## 4. Correlation Heatmap

Shows the Pearson correlation between every pair of numeric features.  
Values range from **-1** (perfect negative) to **+1** (perfect positive).

- **G3 column/row** — which features correlate most with the target?
- **Feature pairs** — highly correlated features carry redundant information.

With G1/G2 removed, the strongest signals expected are `failures` (negative) and parental education `Medu`/`Fedu` (positive).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns

numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

g3_corr = corr['G3'].drop('G3').sort_values(key=abs, ascending=False)
print('=' * 45)
print('  Top features correlated with G3')
print('=' * 45)
for feat, val in g3_corr.items():
    bar = '#' * int(abs(val) * 20)
    sign = '+' if val >= 0 else '-'
    print(f'  {feat:15s}  {sign}{abs(val):.3f}  {bar}')
print('=' * 45)

plt.figure(figsize=(14, 11))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap (G1/G2 removed)', fontsize=13)
plt.tight_layout(); plt.savefig('correlation_heatmap.png', dpi=150); plt.show()
print('  Saved -> correlation_heatmap.png')

## 5. Preprocessing

ML models require all-numeric inputs. Steps:

1. **Label Encoding** — convert categorical text columns to integers
2. **Remove G3 = 0 rows** — students who didn't sit the exam (not true low performers)
3. **Train / Test split (80/20)** — hold out 20% as unseen test data
4. **StandardScaler** — rescale every feature to mean=0, std=1 (critical for KNN, SVR, Ridge, Lasso)

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df_model = df.copy()
le = LabelEncoder()
cat_cols = df_model.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

before = len(df_model)
df_model = df_model[df_model['G3'] > 0].reset_index(drop=True)
removed = before - len(df_model)

X = df_model.drop(columns=['G3'])
y = df_model['G3']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('=' * 45)
print('  Preprocessing Complete')
print('=' * 45)
print(f'  Encoded columns  : {len(cat_cols)}  {cat_cols}')
print(f'  Removed G3=0     : {removed} rows')
print(f'  Remaining rows   : {len(df_model)}')
print(f'  Features         : {X.shape[1]}')
print(f'  Train set        : {X_train_sc.shape[0]} samples')
print(f'  Test  set        : {X_test_sc.shape[0]} samples')
print('=' * 45)

## 6. Model Comparison — 8 Algorithms

We train all 8 models on the same split for a fair comparison.

| Model | Type | Key trait |
|-------|------|-----------|
| Linear Regression | Linear | Baseline |
| Ridge | Linear + L2 | Reduces overfitting |
| Lasso | Linear + L1 | Also does feature selection |
| Decision Tree | Non-linear | Captures interactions, can overfit |
| Random Forest | Ensemble | Reduces DT overfitting via averaging |
| Gradient Boosting | Boosted ensemble | Often the most accurate |
| SVR | Kernel-based | Good on small/high-dim data |
| KNN | Instance-based | Simple, scale-sensitive |

**Metrics:** RMSE (lower=better) - MAE (lower=better) - R2 (higher=better, max 1.0)

In [ ]:
import numpy as np
from sklearn.linear_model  import LinearRegression, Ridge, Lasso
from sklearn.tree          import DecisionTreeRegressor
from sklearn.ensemble      import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm           import SVR
from sklearn.neighbors     import KNeighborsRegressor
from sklearn.metrics       import mean_squared_error, mean_absolute_error, r2_score

MODELS = {
    'Linear Regression' : LinearRegression(),
    'Ridge'             : Ridge(),
    'Lasso'             : Lasso(),
    'Decision Tree'     : DecisionTreeRegressor(random_state=42),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR'               : SVR(),
    'KNN'               : KNeighborsRegressor(),
}

results = {}
print('=' * 60)
print(f"  {'Model':23s}  {'RMSE':>6}  {'MAE':>6}  {'R2':>6}")
print('  ' + '-' * 48)
for name, model in MODELS.items():
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    mae   = mean_absolute_error(y_test, preds)
    r2    = r2_score(y_test, preds)
    results[name] = {'model': model, 'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f"  {name:23s}  {rmse:6.3f}  {mae:6.3f}  {r2:6.3f}")
print('=' * 60)

auto_best_name  = min(results, key=lambda k: results[k]['RMSE'])
auto_best_model = results[auto_best_name]['model']
print(f'\n  Best model (auto): {auto_best_name}')
print(f'     RMSE={results[auto_best_name]["RMSE"]:.3f}   R2={results[auto_best_name]["R2"]:.3f}')

## 7. Model Picker — Choose Your Model

Change `SELECTED_MODEL` below to pick which model to use in Sections 8-10.

```
'auto'               -> auto-pick best (lowest RMSE)
'Linear Regression'  -> simple baseline
'Ridge'              -> regularised linear
'Lasso'              -> regularised + feature selection
'Decision Tree'      -> interpretable, tends to overfit
'Random Forest'      -> robust ensemble
'Gradient Boosting'  -> often most accurate
'SVR'                -> kernel-based
'KNN'                -> nearest-neighbours
```

> This cell is **fully self-contained** — it will re-run Section 6 automatically if you run it before Section 6.

In [ ]:
SELECTED_MODEL = 'auto'

import numpy as np
from sklearn.linear_model  import LinearRegression, Ridge, Lasso
from sklearn.tree          import DecisionTreeRegressor
from sklearn.ensemble      import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm           import SVR
from sklearn.neighbors     import KNeighborsRegressor
from sklearn.metrics       import mean_squared_error, mean_absolute_error, r2_score

if 'results' not in dir() or not results:
    print('  Section 6 was not run — training all models now...')
    _MODELS = {
        'Linear Regression' : LinearRegression(),
        'Ridge'             : Ridge(),
        'Lasso'             : Lasso(),
        'Decision Tree'     : DecisionTreeRegressor(random_state=42),
        'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42),
        'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, random_state=42),
        'SVR'               : SVR(),
        'KNN'               : KNeighborsRegressor(),
    }
    results = {}
    for _name, _model in _MODELS.items():
        _model.fit(X_train_sc, y_train)
        _preds = _model.predict(X_test_sc)
        results[_name] = {
            'model': _model,
            'RMSE' : np.sqrt(mean_squared_error(y_test, _preds)),
            'MAE'  : mean_absolute_error(y_test, _preds),
            'R2'   : r2_score(y_test, _preds),
        }
    auto_best_name  = min(results, key=lambda k: results[k]['RMSE'])
    auto_best_model = results[auto_best_name]['model']
    print(f'  Models trained. Auto-best: {auto_best_name}')

if SELECTED_MODEL == 'auto':
    best_name  = auto_best_name
    best_model = auto_best_model
elif SELECTED_MODEL in results:
    best_name  = SELECTED_MODEL
    best_model = results[SELECTED_MODEL]['model']
else:
    raise ValueError(
        f'Invalid model: "{SELECTED_MODEL}". '
        f'Choose from: {list(results.keys())} or "auto"'
    )

r = results[best_name]
print()
print('=' * 50)
print(f'  Selected Model : {best_name}')
print('  ' + '-' * 46)
print(f'  RMSE  : {r["RMSE"]:.3f}  (avg error in grade points)')
print(f'  MAE   : {r["MAE"]:.3f}')
print(f'  R2    : {r["R2"]:.3f}  ({r["R2"]*100:.1f}% variance explained)')
print('=' * 50)
print('  -> Sections 8, 9, 10 will now use this model.')

## 8. Overfitting Analysis

**Overfitting** = model memorises training data but fails on new data. Two detection methods:

### Method A — Cross-Validation Gap (all 8 models)
Run 5-fold CV on training data -> compare CV RMSE vs hold-out test RMSE.  
- Gap < 0.5 -> generalises well
- Gap > 0.5 -> possible overfitting

### Method B — Learning Curve (selected model only)
Train on increasing fractions of data (10%->100%), plot Train vs Validation RMSE.  
- Curves converge -> good generalisation
- Train RMSE much lower than Val RMSE -> overfitting
- Both curves high -> underfitting (model too simple)

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print('=' * 72)
print(f"  {'Model':23s}  {'CV RMSE':>8}  {'Test RMSE':>10}  {'Gap':>7}  Status")
print('  ' + '-' * 64)
for name, info in results.items():
    cv_scores = cross_val_score(info['model'], X_train_sc, y_train,
                               cv=kf, scoring='neg_root_mean_squared_error')
    cv_rmse   = -cv_scores.mean()
    test_rmse = info['RMSE']
    gap       = test_rmse - cv_rmse
    flag      = 'overfit?' if gap > 0.5 else 'OK'
    marker    = ' <- SELECTED' if name == best_name else ''
    print(f"  {name:23s}  {cv_rmse:8.3f}  {test_rmse:10.3f}  {gap:+7.3f}  {flag}{marker}")
print('=' * 72)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train_sc, y_train,
    cv=5, scoring='neg_root_mean_squared_error',
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1)

train_rmse = -train_scores.mean(axis=1)
val_rmse   = -val_scores.mean(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_rmse, 'o-', color='royalblue', label='Train RMSE')
plt.plot(train_sizes, val_rmse,   's-', color='tomato',    label='Validation RMSE')
plt.fill_between(train_sizes, -train_scores.min(axis=1), -train_scores.max(axis=1), alpha=0.1, color='royalblue')
plt.fill_between(train_sizes, -val_scores.min(axis=1),   -val_scores.max(axis=1),   alpha=0.1, color='tomato')
plt.xlabel('Training set size'); plt.ylabel('RMSE')
plt.title(f'Learning Curve — {best_name}')
plt.legend(); plt.tight_layout()
plt.savefig('learning_curve.png', dpi=150); plt.show()

final_gap = val_rmse[-1] - train_rmse[-1]
print('=' * 45)
print(f'  Learning Curve Summary — {best_name}')
print('  ' + '-' * 41)
print(f'  Train RMSE (full data) : {train_rmse[-1]:.3f}')
print(f'  Val   RMSE (full data) : {val_rmse[-1]:.3f}')
print(f'  Gap                    : {final_gap:+.3f}')
print('  ' + '-' * 41)
if final_gap > 0.5:
    print('  Noticeable gap — possible overfitting.')
    print('     Try reducing complexity or adding regularisation.')
else:
    print('  Gap is acceptable — no significant overfitting.')
print('=' * 45)

## 9. Final Predictions on Test Set

Use the selected model to predict G3 for every student in the hold-out test set.

- Predictions are **rounded** to the nearest integer and **clipped** to [0, 20]
- Two plots:
  - **Actual vs Predicted** — closer to the red line = more accurate
  - **Residual plot** — residuals randomly around 0 = no systematic bias

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

best_model.fit(X_train_sc, y_train)
y_pred         = best_model.predict(X_test_sc)
y_pred_rounded = np.clip(np.round(y_pred).astype(int), 0, 20)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('=' * 50)
print(f'  Final Results — {best_name}')
print('  ' + '-' * 46)
print(f'  RMSE : {rmse:.3f}   (avg error ~ {rmse:.1f} grade points)')
print(f'  MAE  : {mae:.3f}')
print(f'  R2   : {r2:.3f}   ({r2*100:.1f}% variance explained)')
print('=' * 50)

preds_df = pd.DataFrame({
    'Actual G3'    : y_test.values,
    'Predicted G3' : y_pred_rounded,
    'Error'        : y_pred_rounded - y_test.values
}).reset_index(drop=True)

print()
print('  First 20 predictions:')
print('  ' + '-' * 35)
print(preds_df.head(20).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Prediction Results — {best_name}', fontsize=14, fontweight='bold')

axes[0].scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='white', s=60)
mn = min(y_test.min(), y_pred.min()) - 0.5
mx = max(y_test.max(), y_pred.max()) + 0.5
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual G3'); axes[0].set_ylabel('Predicted G3')
axes[0].set_title('Actual vs Predicted'); axes[0].legend()

residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color='tomato', edgecolors='white', s=60)
axes[1].axhline(0, color='black', lw=1.5, linestyle='--')
axes[1].set_xlabel('Predicted G3')
axes[1].set_ylabel('Residual  (Actual - Predicted)')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.savefig('prediction_results.png', dpi=150); plt.show()
print('  Saved -> prediction_results.png')

## 10. Feature Importance

Which features contributed most to the model's predictions?

- **Tree models** (Random Forest, Gradient Boosting) -> `feature_importances_`
- **Linear models** (Ridge, Lasso, Linear Regression) -> absolute coefficient values
- **SVR / KNN** -> no direct importance score (a message will explain this)

In [ ]:
import matplotlib.pyplot as plt, pandas as pd, numpy as np

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
    colors = ['steelblue' if i < 5 else 'lightsteelblue' for i in range(len(importances))]
    plt.figure(figsize=(11, 6))
    importances.plot(kind='bar', color=colors, edgecolor='white')
    plt.title(f'Feature Importances — {best_name}  (top 5 highlighted)')
    plt.ylabel('Importance score'); plt.tight_layout()
    plt.savefig('feature_importances.png', dpi=150); plt.show()
    print('=' * 45)
    print(f'  Feature Importances — {best_name}')
    print('  ' + '-' * 35)
    for i, (feat, val) in enumerate(importances.items()):
        bar = '#' * int(val * 100)
        tag = ' *' if i < 5 else ''
        print(f'  {feat:20s}  {val:.4f}  {bar}{tag}')
    print('=' * 45)

elif hasattr(best_model, 'coef_'):
    coefs = pd.Series(np.abs(best_model.coef_), index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(11, 6))
    coefs.plot(kind='bar', color='steelblue', edgecolor='white')
    plt.title(f'Absolute Coefficients — {best_name}'); plt.ylabel('|Coefficient|')
    plt.tight_layout(); plt.savefig('feature_importances.png', dpi=150); plt.show()
    print('=' * 45)
    print(f'  Coefficients — {best_name}')
    print('  ' + '-' * 35)
    for i, (feat, val) in enumerate(coefs.items()):
        bar = '#' * int(val * 10)
        tag = ' *' if i < 5 else ''
        print(f'  {feat:20s}  {val:.4f}  {bar}{tag}')
    print('=' * 45)

else:
    print('=' * 55)
    print(f'  {best_name} has no importance scores.')
    print('  -> Switch to Random Forest or Gradient Boosting')
    print('     in Section 7 (Model Picker) to see importances.')
    print('=' * 55)